In [0]:
# =============================================================================
# MEMBER 2 — Demand Aggregation & Scoring
# Hengrui Li
# Input:  msbabigdata.spark.trend_market_cleaned_trips  (Member 1 output)
# Output: msbabigdata.spark.trend_market_demand_scores
# =============================================================================
# NOTE ON TIMESTAMPS:
# The cleaned table timestamps have been converted from UTC to NYC time by
# Member 1 (from_utc_timestamp). Because the raw sample data was already in
# NYC local time, the timestamps in the cleaned table are shifted ~5 hours
# earlier than actual pickup times. This does not affect the logic or
# correctness of the pipeline — when the full TLC dataset is loaded, Member 1
# should remove the timezone conversion step, and this notebook requires
# no changes.
# =============================================================================

from pyspark.sql import functions as F
from pyspark.sql import Window

In [0]:
# =============================================================================
# PART 1: Load cleaned trips from Member 1's Delta table
# =============================================================================
 
source_table = "msbabigdata.spark.trend_market_cleaned_trips"
df = spark.table(source_table)
 
# Only select the 4 columns we need — everything else is irrelevant for demand scoring
df = df.select("pulocationid", "pickup_zone", "pickup_borough", "tpep_pickup_datetime")
 
# --- VALIDATION ---
print("=== PART 1: Input Data ===")
print(f"Row count: {df.count()}")
df.show(5)

=== PART 1: Input Data ===
Row count: 2852
+------------+--------------------+--------------+--------------------+
|pulocationid|         pickup_zone|pickup_borough|tpep_pickup_datetime|
+------------+--------------------+--------------+--------------------+
|         231|TriBeCa/Civic Center|     Manhattan| 2022-12-31 19:52:15|
|         231|TriBeCa/Civic Center|     Manhattan| 2022-12-31 19:44:16|
|         231|TriBeCa/Civic Center|     Manhattan| 2022-12-31 19:23:15|
|         231|TriBeCa/Civic Center|     Manhattan| 2022-12-31 19:32:17|
|         231|TriBeCa/Civic Center|     Manhattan| 2022-12-31 19:45:24|
+------------+--------------------+--------------+--------------------+
only showing top 5 rows


In [0]:
# =============================================================================
# PART 2: Feature Extraction — time_bucket + day_type
# =============================================================================
# TIME BUCKETS (4 segments with business meaning):
#   morning_rush : 06–10  commute to work
#   midday       : 10–16  off-peak daytime
#   evening_rush : 16–20  commute home + dining
#   night        : 20–06  late night / overnight
#
# DAY TYPE (2 categories):
#   weekday  : Monday–Friday  (dayofweek 2–6 in Spark, where 1=Sunday)
#   weekend  : Saturday–Sunday (dayofweek 1 or 7)
 
def assign_time_bucket(hour_col):
    return (
        F.when((hour_col >= 6)  & (hour_col < 10), "morning_rush")
         .when((hour_col >= 10) & (hour_col < 16), "midday")
         .when((hour_col >= 16) & (hour_col < 20), "evening_rush")
         .otherwise("night")
    )
 
def assign_day_type(dow_col):
    # Spark dayofweek: 1=Sunday, 2=Monday, ..., 7=Saturday
    return F.when(dow_col.isin(1, 7), "weekend").otherwise("weekday")
 
df_features = (
    df
    .withColumn("hour", F.hour("tpep_pickup_datetime"))
    .withColumn("dow",  F.dayofweek("tpep_pickup_datetime"))
    .withColumn("time_bucket", assign_time_bucket(F.col("hour")))
    .withColumn("day_type",    assign_day_type(F.col("dow")))
)
 
# --- VALIDATION ---
print("\n=== PART 2: Feature Extraction ===")
print("Time bucket distribution:")
df_features.groupBy("time_bucket").count().orderBy("count", ascending=False).show()
print("Day type distribution:")
df_features.groupBy("day_type").count().orderBy("count", ascending=False).show()


=== PART 2: Feature Extraction ===
Time bucket distribution:
+------------+-----+
| time_bucket|count|
+------------+-----+
|evening_rush| 2846|
|       night|    6|
+------------+-----+

Day type distribution:
+--------+-----+
|day_type|count|
+--------+-----+
| weekday| 1889|
| weekend|  963|
+--------+-----+



In [0]:
# =============================================================================
# PART 3: Zone Aggregation — trip_count per (zone × time_bucket × day_type)
# =============================================================================
 
df_agg = (
    df_features
    .groupBy("pulocationid", "pickup_zone", "pickup_borough", "time_bucket", "day_type")
    .agg(
        F.count("*").alias("trip_count"),
        F.countDistinct(F.to_date("tpep_pickup_datetime")).alias("days_observed")
    )
)
 
# --- VALIDATION ---
print("\n=== PART 3: Zone Aggregation ===")
print(f"Aggregated rows (zone × time_bucket × day_type cells): {df_agg.count()}")
print("Sample — top zones by trip count:")
df_agg.orderBy("trip_count", ascending=False).show(10)
 


=== PART 3: Zone Aggregation ===
Aggregated rows (zone × time_bucket × day_type cells): 143
Sample — top zones by trip count:
+------------+--------------------+--------------+------------+--------+----------+-------------+
|pulocationid|         pickup_zone|pickup_borough| time_bucket|day_type|trip_count|days_observed|
+------------+--------------------+--------------+------------+--------+----------+-------------+
|         132|         JFK Airport|        Queens|evening_rush| weekday|       164|            2|
|         230|Times Sq/Theatre ...|     Manhattan|evening_rush| weekday|        98|            2|
|          79|        East Village|     Manhattan|evening_rush| weekday|        88|            2|
|         249|        West Village|     Manhattan|evening_rush| weekday|        81|            2|
|         114|Greenwich Village...|     Manhattan|evening_rush| weekday|        78|            2|
|         234|            Union Sq|     Manhattan|evening_rush| weekday|        76|      

In [0]:
# =============================================================================
# PART 4: Demand Score Computation
# =============================================================================
# demand_score = trip_count / avg(trip_count across all zones for same time window)
#
# score > 1.0  →  above average demand for this time window
# score = 1.5  →  50% above average
# score < 1.0  →  below average
#
# We use a Window partitioned by (time_bucket, day_type) so the comparison is
# always apples-to-apples: a zone is only compared against other zones in the
# same time window, not against the global average.
 
window_spec = Window.partitionBy("time_bucket", "day_type")
 
df_scored = (
    df_agg
    .withColumn("avg_trip_count", F.avg("trip_count").over(window_spec))
    .withColumn("stddev_trip_count", F.stddev("trip_count").over(window_spec))
    .withColumn(
        "demand_score",
        F.round(F.col("trip_count") / F.col("avg_trip_count"), 4)
    )
)
 
# --- VALIDATION ---
print("\n=== PART 4: Demand Scores ===")
print("Score distribution summary:")
df_scored.select("demand_score").summary().show()
print("Top 10 highest-scoring zones:")
df_scored.orderBy("demand_score", ascending=False).show(10)


=== PART 4: Demand Scores ===
Score distribution summary:
+-------+------------------+
|summary|      demand_score|
+-------+------------------+
|  count|               143|
|   mean|0.9999951048951061|
| stddev|1.0516073723971762|
|    min|            0.0403|
|    25%|             0.127|
|    50%|            0.7617|
|    75%|            1.5234|
|    max|            6.6122|
+-------+------------------+

Top 10 highest-scoring zones:
+------------+--------------------+--------------+------------+--------+----------+-------------+------------------+------------------+------------+
|pulocationid|         pickup_zone|pickup_borough| time_bucket|day_type|trip_count|days_observed|    avg_trip_count| stddev_trip_count|demand_score|
+------------+--------------------+--------------+------------+--------+----------+-------------+------------------+------------------+------------+
|         132|         JFK Airport|        Queens|evening_rush| weekday|       164|            2| 24.80263157894737

In [0]:
 
# =============================================================================
# PART 5: Low-Confidence Flagging
# =============================================================================
# A zone-time cell is flagged as low_confidence if ANY of:
#   (a) trip_count < 3      — too few trips, pattern not reliable
#   (b) days_observed < 2   — only seen on 1 day, could be an anomaly
#   (c) CoV > 2.0           — coefficient of variation too high (stddev / mean),
#                             demand is too erratic to trust
#                             CoV = stddev / avg across zones in same window
#
# NOTE ON THRESHOLDS:
# These thresholds are intentionally low because the sample data is small.
# With full TLC data (millions of rows), raise trip_count threshold to ~20
# and days_observed to ~10 for more conservative filtering.
 
TRIP_COUNT_MIN  = 3
DAYS_OBSERVED_MIN = 2
COV_MAX = 2.0
 
df_final = (
    df_scored
    .withColumn(
        "coeff_of_variation",
        F.round(
            F.when(
                F.col("avg_trip_count") > 0,
                F.col("stddev_trip_count") / F.col("avg_trip_count")
            ).otherwise(None),
            4
        )
    )
    .withColumn(
        "is_low_confidence",
        (F.col("trip_count") < TRIP_COUNT_MIN) |
        (F.col("days_observed") < DAYS_OBSERVED_MIN) |
        (F.col("coeff_of_variation") > COV_MAX)
    )
    # Clean up intermediate columns not needed downstream
    .drop("stddev_trip_count")
)
 
# --- VALIDATION ---
print("\n=== PART 5: Low-Confidence Flagging ===")
flag_counts = df_final.groupBy("is_low_confidence").count()
flag_counts.show()
 
total = df_final.count()
low_conf = df_final.filter(F.col("is_low_confidence") == True).count()
print(f"Low-confidence cells: {low_conf} / {total} ({low_conf/total*100:.1f}%)")
print(f"High-confidence cells available for ranking: {total - low_conf}")
 
print("\nSample of flagged zones:")
df_final.filter(F.col("is_low_confidence") == True) \
        .select("pickup_zone", "time_bucket", "day_type", "trip_count", "days_observed", "demand_score", "is_low_confidence") \
        .show(5)
 
print("\nSample of high-confidence zones:")
df_final.filter(F.col("is_low_confidence") == False) \
        .orderBy("demand_score", ascending=False) \
        .select("pickup_zone", "pickup_borough", "time_bucket", "day_type", "trip_count", "demand_score", "is_low_confidence") \
        .show(10)


=== PART 5: Low-Confidence Flagging ===
+-----------------+-----+
|is_low_confidence|count|
+-----------------+-----+
|            false|   52|
|             true|   91|
+-----------------+-----+

Low-confidence cells: 91 / 143 (63.6%)
High-confidence cells available for ranking: 52

Sample of flagged zones:
+--------------------+------------+--------+----------+-------------+------------+-----------------+
|         pickup_zone| time_bucket|day_type|trip_count|days_observed|demand_score|is_low_confidence|
+--------------------+------------+--------+----------+-------------+------------+-----------------+
|Central Harlem North|evening_rush| weekday|         1|            1|      0.0403|             true|
|       South Jamaica|evening_rush| weekday|         1|            1|      0.0403|             true|
|  Claremont/Bathgate|evening_rush| weekday|         1|            1|      0.0403|             true|
|       East New York|evening_rush| weekday|         1|            1|      0.0403| 

In [0]:
# =============================================================================
# PART 6: Write to Delta Lake
# =============================================================================
# Output schema for Member 3 (Heuristic Ranking):
#   pulocationid        — zone ID (join key)
#   pickup_zone         — human-readable zone name
#   pickup_borough      — borough name
#   time_bucket         — morning_rush / midday / evening_rush / night
#   day_type            — weekday / weekend
#   trip_count          — raw pickup count in this cell
#   days_observed       — number of distinct days this cell was observed
#   avg_trip_count      — citywide average for this time window
#   demand_score        — trip_count / avg_trip_count (>1.0 = above average)
#   coeff_of_variation  — stddev/mean across zones in window (signal reliability)
#   is_low_confidence   — boolean flag for Member 3 to filter out
 
target_table = "msbabigdata.spark.trend_market_demand_scores"
 
(
    df_final
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(target_table)
)
 
# --- VALIDATION ---
print("\n=== PART 6: Delta Lake Write Validation ===")
df_check = spark.table(target_table)
print(f"Rows written to {target_table}: {df_check.count()}")
print("\nFinal table schema:")
df_check.printSchema()
print("\nFinal table sample (top demand zones, high confidence only):")
display(
    spark.sql(f"""
        SELECT pickup_zone, pickup_borough, time_bucket, day_type,
               trip_count, demand_score, is_low_confidence
        FROM {target_table}
        WHERE is_low_confidence = false
        ORDER BY demand_score DESC
        LIMIT 15
    """)
)
 